![NVIDIA Header](../../shared/assets/nvidia_header.png)

# NASA APOD: Your First API-backed Skill

### Review a canonical skill, add its network policy, deploy it, and harden its response boundary

⏱ Estimated time: 25 minutes

## Learning Objectives

By the end of this notebook, you will be able to:

<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Validate the frontmatter and bounded commands in a canonical <a href="https://docs.openclaw.ai/tools/creating-skills">SKILL.md</a></p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Add a targeted <a href="https://docs.nvidia.com/openshell/reference/policy-schema">network policy</a> for one API endpoint</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Deploy canonical skill content to one verified <a href="https://docs.openclaw.ai/concepts/agent-workspace">agent workspace</a></p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Distinguish skill, policy, deployment, and live-response evidence</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Independently harden the APOD response boundary with deterministic validation</p>

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2024-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

## Setup

**Platform:** Validated NemoClaw host (Ubuntu 24.04, DGX Spark, or Apple Silicon macOS with Colima)

**Prerequisites:** Complete **[NemoClaw with OpenClaw](../03-nemoclaw/01_nemoclaw_openclaw.ipynb)** and confirm the exact `agent` sandbox before changing policy or deploying skills.

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
for _parent in [_cwd, *_cwd.parents]:
    if (_parent / "pyproject.toml").exists():
        if str(_parent) not in sys.path:
            sys.path.insert(0, str(_parent))
        break

In [ ]:
from shared.utils.setup import setup

setup(
    require_docker=True,
    require_node=True,
    require_openshell=True,
    required_keys=[],
)

In [ ]:
import importlib
import json
import shlex
import time
from datetime import date
from pathlib import Path
from urllib.parse import urlsplit

import shared.utils.platform as _plat_mod

importlib.reload(_plat_mod)

from shared.utils.display import run_command
from shared.utils.mutations import authorize_mutation
from shared.utils.platform import (
    create_and_onboard_agent,
    deploy_skill_to_agent,
    get_dashboard_access_note,
    get_dashboard_access_url,
    list_openclaw_agent_ids,
)

MUTATION_SANDBOX = "agent"

nasa_soul_path = Path("souls/nasa-explorer.md")
if not nasa_soul_path.is_file():
    raise FileNotFoundError(f"Canonical NASA SOUL is missing: {nasa_soul_path}")
nasa_soul = nasa_soul_path.read_text(encoding="utf-8")

nasa_agent_preexisting = "nasa-apod" in list_openclaw_agent_ids(MUTATION_SANDBOX)
nasa_agent_ready = False
course_agent_created = False
agent_reconciliation_allowed = nasa_agent_preexisting
if not agent_reconciliation_allowed:
    agent_reconciliation_allowed = authorize_mutation(
        action="Create the course-owned nasa-apod agent from its canonical SOUL.md",
        target=MUTATION_SANDBOX,
    )

if agent_reconciliation_allowed:
    creation_ok, detail = create_and_onboard_agent(
        "nasa-apod",
        name="NASA Explorer",
        emoji="🔭",
        soul_md=nasa_soul,
        sandbox=MUTATION_SANDBOX,
        update_existing=False,
    )
    print(f"Agent: {detail}")
    nasa_agent_ready = creation_ok
    course_agent_created = creation_ok and not nasa_agent_preexisting
    if not creation_ok:
        print("SAFE SKIP: existing or partial agent state was preserved; later deployment skips.")
else:
    print("SAFE MODE: nasa-apod was absent and was not created.")

dashboard_url = get_dashboard_access_url()
dashboard_note = get_dashboard_access_note()
print(f"Dashboard: {dashboard_url}")
print(dashboard_note)

# The notebook probe and deployed skill intentionally use DEMO_KEY. A key in
# the host .env is not automatically available to an agent inside the sandbox.
NASA_APOD_DEMO_KEY = "DEMO_KEY"
print("NASA APOD credential scope: public DEMO_KEY (not host .env state)")


def nasa_apod_url(extra_query: str = "") -> str:
    suffix = f"&{extra_query}" if extra_query else ""
    return f"https://api.nasa.gov/planetary/apod?api_key={NASA_APOD_DEMO_KEY}{suffix}"


def nasa_apod_curl_command(
    *, timeout: int, fail_fast: bool = True, extra_query: str = ""
) -> tuple[str, str]:
    flags = "-4 -fsS" if fail_fast else "-4 -sS"
    url = shlex.quote(nasa_apod_url(extra_query))
    command = (
        f"openshell sandbox exec -n {MUTATION_SANDBOX} --timeout {timeout} "
        f"-- curl {flags} --max-time {timeout} {url}"
    )
    return command, command


def _live_apod_payload_is_usable(payload: object) -> bool:
    if not isinstance(payload, dict):
        return False
    required_text = ("date", "title", "explanation", "media_type", "url")
    if any(
        not isinstance(payload.get(field), str) or not payload[field].strip()
        for field in required_text
    ):
        return False
    try:
        date.fromisoformat(payload["date"])
    except ValueError:
        return False
    if payload["media_type"] not in {"image", "video"}:
        return False
    media_urls = [payload["url"]]
    if payload.get("hdurl") is not None:
        if not isinstance(payload["hdurl"], str) or not payload["hdurl"].strip():
            return False
        media_urls.append(payload["hdurl"])
    return all(urlsplit(url).scheme == "https" and bool(urlsplit(url).netloc) for url in media_urls)


def run_nasa_apod_request(
    *,
    extra_query: str = "",
    timeout: int = 45,
    attempts: int = 4,
    pause: int = 5,
    label: str = "NASA APOD request",
) -> dict:
    """Run a live APOD request with bounded retries and shape validation."""
    last_output = ""
    for attempt in range(1, attempts + 1):
        actual_cmd, display_cmd = nasa_apod_curl_command(
            timeout=timeout,
            extra_query=extra_query,
        )
        print(f"{label}: attempt {attempt}/{attempts}")
        result = run_command(
            actual_cmd,
            display_cmd=display_cmd,
            timeout=timeout + 15,
            format_json=True,
            return_result=True,
        )
        last_output = f"{result.stdout}\n{result.stderr}".strip()
        if result.returncode == 0:
            try:
                payload = json.loads(result.stdout)
            except json.JSONDecodeError:
                print("APOD response was not valid JSON; retrying.")
            else:
                if _live_apod_payload_is_usable(payload):
                    print(
                        "APOD_OK "
                        f"date={payload['date']} "
                        f"title={payload['title']} "
                        f"media_type={payload['media_type']}"
                    )
                    return payload
                print("APOD JSON failed the hardened response contract; retrying.")
        elif "503" in last_output:
            print("NASA APOD returned 503; retrying live-service request.")
        else:
            print(f"APOD request failed with exit code {result.returncode}; retrying.")
        if attempt < attempts:
            time.sleep(pause)
    raise RuntimeError(
        f"{label} did not return usable APOD JSON after {attempts} attempts. "
        f"Last output:\n{last_output[:1000]}"
    )

---

## 1. What We're Building

A **[skill](https://docs.openclaw.ai/tools/creating-skills)** is a self-contained instruction file that teaches an OpenClaw agent how to use a specific tool or API. Without a skill, the agent doesn't know that NASA's APOD endpoint exists or how to call it — skills extend an agent's capabilities beyond its built-in knowledge.

You'll inspect and deploy the checked-in skill for NASA's **Astronomy Picture of the Day** (APOD). Every integration follows the same workflow:

```text
Step 1: Inspect canonical files  → Review the exact SOUL.md and SKILL.md sources
Step 2: Add the network policy  → Allow the sandbox to reach the API
Step 3: Deploy the skill        → Install through OpenClaw for one exact agent
Step 4: Test it                 → Validate the live response before presenting it
```

**Why NASA APOD?** It's a compact real integration:

- **Low-friction auth** — the checked-in lesson and deployed skill both use the public, rate-limited `DEMO_KEY`
- **One endpoint** — `api.nasa.gov/planetary/apod`
- **One HTTP method** — `GET`
- **Visually impressive** — returns astronomy media with explanations

A key in the notebook host's `.env` is deliberately out of scope: it is not automatically injected into the sandbox agent. Notebooks 02 and 03 apply the same pattern to more APIs and custom personalities.

---

## 2. Inspect the Canonical SKILL.md

Every skill lives in its own directory and is defined by [`SKILL.md`](https://docs.openclaw.ai/tools/creating-skills). The checked-in file is the source of truth; running the notebook must not regenerate or silently change it.

OpenClaw requires the YAML frontmatter delimiter on line 1. The SPDX comment therefore follows the closing frontmatter delimiter, before the instructional Markdown body. The frontmatter tells OpenClaw *whether* to load the skill; the body tells the agent *how* to use it.

In [ ]:
from pathlib import Path

nasa_skill_path = Path("skills/nasa-apod/SKILL.md")
assert nasa_skill_path.is_file(), f"Missing canonical skill: {nasa_skill_path}"
print(f"Canonical skill found: {nasa_skill_path}")

Read and validate the canonical NASA skill without modifying it. **Expected result:** the cell confirms line-1 frontmatter, the SPDX placement, and then displays the exact checked-in source. This keeps notebook execution reproducible and prevents source drift.

In [ ]:
nasa_skill_text = nasa_skill_path.read_text(encoding="utf-8")
nasa_skill_lines = nasa_skill_text.splitlines()
assert nasa_skill_lines[0] == "---", "SKILL.md frontmatter must begin on line 1"
closing_frontmatter = nasa_skill_lines.index("---", 1)
assert nasa_skill_lines[closing_frontmatter + 1] == "<!--"
assert "SPDX-License-Identifier: Apache-2.0" in "\n".join(
    nasa_skill_lines[closing_frontmatter + 1 : closing_frontmatter + 7]
)
print(nasa_skill_text)

**What the canonical file contains:**

- **Frontmatter `description`** — tells the LLM when the skill applies and sets clear exclusions.
- **`requires.bins: ["curl"]`** — keeps the skill ineligible when its HTTP client is missing.
- **Body** — bounded API calls, response requirements, presentation rules, and error handling.
- **Credential boundary** — the deployed skill uses `DEMO_KEY`; it does not claim access to a host `.env` file.

### The Full SKILL.md Spec

```yaml
---
name: my-skill                    # Required: unique identifier
description: "When to use..."     # Required: trigger phrases + scope
homepage: https://example.com     # Optional: project URL
user-invocable: true              # Optional: can user trigger via /my-skill?
metadata:
  openclaw:
    emoji: "🔧"                   # Optional: display emoji
    requires:
      bins: ["curl"]              # Required binaries
      env: ["API_KEY"]            # Required environment, when configured
    os: ["linux", "darwin"]       # Supported platforms
---
```

A skill directory can also include optional supporting directories:

```text
my-skill/
├── SKILL.md          # Required: frontmatter + instructions
├── scripts/          # Optional: helper scripts
├── references/       # Optional: reference docs
└── assets/           # Optional: example output and templates
```

### Skill Discovery Hierarchy

OpenClaw 2026.6.10 checks these roots in highest-to-lowest precedence order:

| Priority | Location | Visibility |
|----------|----------|------------|
| 1 | `<workspace>/skills/` | One agent workspace |
| 2 | `<workspace>/.agents/skills/` | One project-agent workspace |
| 3 | `~/.agents/skills/` | Personal skills visible to local agents |
| 4 | `~/.openclaw/skills/` | Shared managed skills |
| 5 | Bundled skills | Shipped with the installed OpenClaw build |
| 6 | Extra directories and plugin skills | Lowest-precedence configured sources |

Same-name skills resolve to the highest-precedence source. The number of bundled skills varies by build, so the lesson intentionally does not hardcode it.

### Independent Exercise: Harden the APOD Response Boundary

A successful HTTP status does not guarantee a usable APOD document. Implement
`validate_apod_payload()` so the skill accepts well-formed image or video records
and rejects malformed dates, non-HTTPS media URLs, missing text, and API error
documents.

Work only with the supplied Python objects. The exercise never calls NASA or
changes the sandbox policy.

**Reasoning note:** In the course worksheet or LMS response for
`nasa-response-hardening`, record 1–2 sentences explaining why both URLs belong
inside the trust boundary, what evidence supports your answer, and what response
would falsify it.


In [ ]:
# FIXME: implement all checks described in the exercise prompt.
def validate_apod_payload(payload):
    return False

Run the validator against deterministic image, video, and malformed fixtures.
**Expected result:** after your function handles every boundary, the cell prints
`NASA_RESPONSE_HARDENING_OK`. No network request is made.

In [ ]:
valid_image = {
    "date": "2026-01-15",
    "title": "A Test Nebula",
    "explanation": "A deterministic fixture used for validation.",
    "media_type": "image",
    "url": "https://example.test/nebula.jpg",
    "hdurl": "https://example.test/nebula-hd.jpg",
}
valid_video = {
    "date": "2026-01-16",
    "title": "A Test Flyby",
    "explanation": "A deterministic video fixture.",
    "media_type": "video",
    "url": "https://example.test/flyby",
}

cases = [
    (valid_image, True, "valid image"),
    (valid_video, True, "valid video"),
    (None, False, "non-dictionary payload"),
    ({**valid_image, "date": "15-01-2026"}, False, "non-ISO date"),
    ({**valid_image, "url": "http://example.test/nebula.jpg"}, False, "non-HTTPS URL"),
    (
        {key: value for key, value in valid_image.items() if key != "explanation"},
        False,
        "missing explanation",
    ),
    ({**valid_image, "title": ""}, False, "empty title"),
    ({**valid_image, "hdurl": ""}, False, "empty HD URL"),
    (
        {**valid_image, "hdurl": "http://example.test/nebula-hd.jpg"},
        False,
        "non-HTTPS HD URL",
    ),
    ({**valid_image, "media_type": "binary"}, False, "unknown media type"),
    ({"error": {"message": "rate limited"}}, False, "API error document"),
]

observed = [validate_apod_payload(payload) for payload, _, _ in cases]
expected = [expected_value for _, expected_value, _ in cases]
assert observed == expected, [
    label
    for result, (_, expected_value, label) in zip(observed, cases, strict=True)
    if result != expected_value
]
print("NASA_RESPONSE_HARDENING_OK")

<details>
<summary>Solution</summary>

```python
from datetime import date
from urllib.parse import urlsplit


def validate_apod_payload(payload):
    if not isinstance(payload, dict):
        return False
    required_text = ("date", "title", "explanation", "media_type", "url")
    if any(not isinstance(payload.get(field), str) or not payload[field].strip() for field in required_text):
        return False
    try:
        date.fromisoformat(payload["date"])
    except ValueError:
        return False
    if payload["media_type"] not in {"image", "video"}:
        return False
    media_urls = [payload["url"]]
    if payload.get("hdurl") is not None:
        if not isinstance(payload["hdurl"], str) or not payload["hdurl"].strip():
            return False
        media_urls.append(payload["hdurl"])
    return all(
        urlsplit(url).scheme == "https" and bool(urlsplit(url).netloc)
        for url in media_urls
    )
```

</details>

In [ ]:
# # Fallback solution: uncomment these lines only after attempting the exercise.
# from datetime import date
# from urllib.parse import urlsplit
#
#
# def validate_apod_payload(payload):
#     if not isinstance(payload, dict):
#         return False
#     required_text = ("date", "title", "explanation", "media_type", "url")
#     if any(not isinstance(payload.get(field), str) or not payload[field].strip() for field in required_text):
#         return False
#     try:
#         date.fromisoformat(payload["date"])
#     except ValueError:
#         return False
#     if payload["media_type"] not in {"image", "video"}:
#         return False
#     media_urls = [payload["url"]]
#     if payload.get("hdurl") is not None:
#         if not isinstance(payload["hdurl"], str) or not payload["hdurl"].strip():
#             return False
#         media_urls.append(payload["hdurl"])
#     return all(
#         urlsplit(url).scheme == "https" and bool(urlsplit(url).netloc)
#         for url in media_urls
#     )

---

## 3. Add the Network Policy

OpenShell [sandboxes](https://docs.nvidia.com/openshell/sandboxes/manage-sandboxes)
run in a **zero-trust networking environment**: outbound access requires an explicit
[network policy](https://docs.nvidia.com/openshell/reference/policy-schema). The
egress proxy evaluates host, port, method, and path before a request leaves the
sandbox.

The next cells are ownership-safe. They inspect any existing `nasa_apod` entry and
preserve it unchanged. A blocked-before/allowed-after demonstration runs only when
that entry is absent; adding the course rule then requires explicit target-scoped
mutation authorization. Safe mode never removes or overwrites someone else's rule.


In [ ]:
from shared.utils.policies import load_base_policy

current_policy = load_base_policy(MUTATION_SANDBOX)
existing_nasa_policy = current_policy.get("network_policies", {}).get("nasa_apod")
nasa_policy_preexisting = existing_nasa_policy is not None
nasa_policy_ready = False
blocked_proof_observed = False
course_policy_added = False

if nasa_policy_preexisting:
    print("PRESERVED: an existing nasa_apod policy entry is already present.")
    print("The blocked-before step is skipped because removing it would destroy prior state.")
else:
    print("No existing nasa_apod entry; running a read-only default-deny probe.")
    actual_cmd, display_cmd = nasa_apod_curl_command(timeout=10, fail_fast=False)
    blocked_result = run_command(
        actual_cmd,
        display_cmd=display_cmd,
        timeout=15,
        return_result=True,
    )
    blocked_output = f"{blocked_result.stdout}\n{blocked_result.stderr}"
    assert blocked_result.returncode != 0 and "403" in blocked_output, blocked_output
    blocked_proof_observed = True
    print("Blocked as expected: CONNECT proxy returned 403 with no APOD rule.")

Add the narrow APOD endpoint only when the course owns the missing entry and the
target-scoped mutation is explicitly authorized. If an entry already exists, the
cell preserves and reports it instead of overwriting it. In safe mode, it prints the
opt-in and leaves the policy unchanged.


In [ ]:
from shared.utils.policies import upsert_network_policies

nasa_apod_policy = {
    "nasa_apod": {
        "name": "nasa_apod",
        "endpoints": [
            {
                "host": "api.nasa.gov",
                "port": 443,
                "protocol": "rest",
                "enforcement": "enforce",
                "rules": [{"allow": {"method": "GET", "path": "/planetary/apod"}}],
            }
        ],
        "binaries": [
            {"path": "/usr/bin/curl"},
            {"path": "/usr/local/bin/node"},
        ],
    }
}


def apod_entry_supports_course_request(entry: object) -> bool:
    """Accept only the exact course entry; reject capability supersets."""
    return entry == nasa_apod_policy["nasa_apod"]


# Reload immediately before the decision so stale state cannot authorize overwrite.
policy = load_base_policy(MUTATION_SANDBOX)
fresh_existing = policy.get("network_policies", {}).get("nasa_apod")
if fresh_existing is not None:
    nasa_policy_preexisting = True
    nasa_policy_ready = apod_entry_supports_course_request(fresh_existing)
    print("PRESERVED: fresh policy read found nasa_apod; it was not changed.")
    if not nasa_policy_ready:
        print("SAFE SKIP: existing entry is not the exact narrow course shape.")
elif authorize_mutation(
    action="Add the course-owned nasa_apod network policy",
    target=MUTATION_SANDBOX,
):
    try:
        policy = upsert_network_policies(
            MUTATION_SANDBOX,
            nasa_apod_policy,
            timeout=30,
            replace_existing=False,
        )
    except RuntimeError:
        policy = load_base_policy(MUTATION_SANDBOX)
        raced_entry = policy.get("network_policies", {}).get("nasa_apod")
        if raced_entry is None:
            raise
        nasa_policy_preexisting = True
        nasa_policy_ready = apod_entry_supports_course_request(raced_entry)
        print("PRESERVED: nasa_apod appeared before apply; no replacement occurred.")
    else:
        applied_entry = policy.get("network_policies", {}).get("nasa_apod")
        nasa_policy_ready = apod_entry_supports_course_request(applied_entry)
        if not nasa_policy_ready:
            raise RuntimeError("Applied APOD policy did not retain the exact narrow shape")
        course_policy_added = True
        print("ADDED: course-owned nasa_apod policy entry")
else:
    print("SAFE MODE: no nasa_apod rule was added; live APOD probes will be skipped.")

print(f"APOD policy ready for the course request: {nasa_policy_ready}")

When a usable APOD rule was pre-existing or explicitly added, repeat the request.
**Expected result:** the probe receives usable APOD JSON. When safe mode left the
rule absent, the cell reports a safe skip rather than presenting a failed request as
proof. Only a course-owned addition supports a blocked-before/allowed-after claim.


In [ ]:
# Require a usable response only when policy readiness was established.
post_policy_payload = None
if nasa_policy_ready:
    time.sleep(3)
    print("Testing api.nasa.gov through the currently provisioned APOD rule:")
    print(
        "Force IPv4 because the sandbox proxy path to api.nasa.gov is more "
        "reliable that way on the verified Spark setup."
    )
    post_policy_payload = run_nasa_apod_request(label="Post-policy APOD probe")
else:
    print("SAFE SKIP: no APOD rule was pre-existing or explicitly added.")

If `post_policy_payload` contains a validated object, the live API returned usable APOD data through the currently provisioned rule. Network-policy updates are hot-reloadable, so an authorized course-owned addition did not require sandbox recreation. A preserved incompatible entry or a safe-mode skip produces no live-success claim.

---

## 4. Deploy the Skill to the NASA Explorer Agent

OpenClaw 2026.6.10 supports exact-agent local-directory installs. When a source directory is already inside the sandbox, the native form is:

```bash
openshell sandbox exec -n agent --no-tty -- \
  openclaw skills install /sandbox/path/to/nasa-apod --agent nasa-apod
```

`--agent nasa-apod` resolves that agent's workspace; shared installation requires an explicit `--global` instead. The course helper below uploads the checked-in directory to a unique staging path, invokes this supported exact-agent install surface, compares existing content, verifies the result, and removes staging state.

In [ ]:
# Deploy only to a verified agent and only after target-scoped authorization.
skill_ready = False
if not nasa_agent_ready:
    print("SAFE SKIP: nasa-apod profile did not match; no skill was deployed.")
elif authorize_mutation(
    action="Deploy or preserve the nasa-apod exact-agent skill",
    target=MUTATION_SANDBOX,
):
    ok, detail = deploy_skill_to_agent(
        "nasa-apod",
        "skills/nasa-apod/",
        sandbox=MUTATION_SANDBOX,
        overwrite_existing=False,
        remove_global=False,
    )
    print(detail)
    if ok:
        skill_ready = True
    else:
        print("SAFE SKIP: conflicting workspace skill was preserved; readiness is false.")
else:
    print("SAFE MODE: workspace and global skills were unchanged.")

Reconcile the workspace-scoped skill outcome. **Expected result:** an authorized new
deployment succeeds, an existing workspace skill is preserved, or safe mode reports
that no deployment ran. The helper never removes a global skill implicitly.


In [ ]:
if skill_ready:
    print("SKILL_SCOPE_READY: nasa-apod workspace skill is available or preserved.")
else:
    print("SAFE SKIP: no workspace-skill readiness claim on this execution path.")

---

## 5. Test the API Contract

When `nasa_policy_ready` is true, call APOD from the exact sandbox and accept only HTTPS media responses that pass the same hardened shape contract taught in the exercise. Skill deployment and API-policy readiness remain separate evidence.

### Today's Picture

In [ ]:
# Today's Astronomy Picture of the Day, only when policy readiness is real.
today_payload = None
if nasa_policy_ready:
    today_payload = run_nasa_apod_request(label="Today's APOD")
else:
    print("SAFE SKIP: today's APOD request needs a pre-existing or authorized rule.")

You should see a JSON response with `title`, `explanation`, `url`, and `hdurl`.
**Click the `hdurl` link** to see the full-resolution astronomy image.

### Historical Date Example

In [ ]:
# Historical APOD example: Christmas 2025, only when policy readiness is real.
historical_payload = None
if nasa_policy_ready:
    historical_payload = run_nasa_apod_request(
        extra_query="date=2025-12-25",
        label="Historical APOD (2025-12-25)",
    )
else:
    print("SAFE SKIP: historical APOD request needs an available APOD rule.")

---

## 6. Try It Live

Run this optional interaction only when `nasa_agent_ready`, `skill_ready`, and `nasa_policy_ready` are all true. Resetting a session is a host-side NemoClaw lifecycle operation, not an in-chat `/new` message:

```bash
# Optional: replace <session-key> with the exact key shown in Sessions.
nemoclaw agent sessions reset '<session-key>' --agent nasa-apod

# Send the prompt to the exact agent inside the exact sandbox.
openshell sandbox exec -n agent --no-tty -- \
  openclaw agent --agent nasa-apod --message "What is today's Astronomy Picture of the Day?"
```

Open the dashboard URL printed earlier. On DGX Spark, use the tokenized root URL over the workstation SSH tunnel. Inspect the resulting entry in the Dashboard **Sessions** view.

**Try these prompts:**

- *"What's today's Astronomy Picture of the Day?"*
- *"Show me the NASA photo from last Christmas"*
- *"Show me the NASA photo from Christmas 2025"*

> **Why `--agent nasa-apod`?** The skill is installed only for that agent's workspace. Omitting `--agent` can route to another agent that cannot see it.

---

## 7. What Just Happened

```text
1. OpenClaw verifies SOUL.md       → Exact canonical NASA Explorer persona
2. OpenClaw discovers SKILL.md    → Bounded DEMO_KEY APOD instructions
3. User asks about APOD           → Agent selects the nasa-apod skill
4. Agent constructs bounded curl  → HTTPS GET with fail-fast and max-time flags
5. Sandbox runs the request       → Traffic traverses the egress proxy
6. Proxy evaluates policy         → Exact GET /planetary/apod capability
7. NASA returns JSON              → Response must pass the hardened validator
8. Agent formats the result       → Title, summary, media link, and credit
```

If any piece is missing, the notebook preserves state and refuses to claim the full round trip. A skill teaches behavior, policy controls egress, and deployment makes the skill visible to one agent; these are separate controls.

---

## Real-World Connection

The NASA APOD demo is useful because it shows several NVIDIA agent-runtime ideas in one small flow:

- **A visible outcome** — astronomy media makes successful response handling easy to inspect.
- **Explicit credential scope** — the deployed skill uses `DEMO_KEY`; host `.env` state is not implied.
- **Layered control** — SOUL instructions, exact-agent skill installation, and OpenShell egress policy each have a distinct role.
- **Fail-closed evidence** — malformed JSON, incompatible policies, and skill conflicts do not become success claims.

The next notebook builds a Daily Financial Briefing agent that composes five public APIs with a canonical financial-analyst persona.

---

## Checkpoint

Verify only the outcomes your execution path actually produced.

1. **Check:** the exact agent profile was verified.
   - **Expected:** `create_and_onboard_agent` reports that the identity, workspace, and canonical SOUL already match, or creates and verifies the missing agent after authorization.
   - **Safe mode/conflict:** `nasa_agent_ready` stays false and later deployment skips.

2. **Check:** policy ownership remained safe.
   - **Expected:** an exact existing entry was preserved, or a missing entry was added without replacement after authorization.
   - **Safe mode/conflict:** no entry was changed and live probes were skipped.

3. **Check:** skill ownership remained safe.
   - **Expected:** identical content was preserved or the canonical skill was installed for `nasa-apod` after authorization.
   - **Conflict:** `skill_ready` remains false; differing workspace content is preserved.

4. **Check:** your independent response validator handled every static fixture.
   - **Expected:** `NASA_RESPONSE_HARDENING_OK` prints without network access.
   - **If not working:** inspect non-dictionary, missing-explanation, and unsafe-URL cases.

---

## ✅ What You Accomplished

<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Inspected the canonical SOUL.md and SKILL.md without rewriting either file</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Verified or safely preserved the exact agent profile before deployment</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Rejected APOD policy supersets instead of treating broader access as equivalent</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Kept the deployed skill on DEMO_KEY rather than implying access to host secrets</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Distinguished live proof, preserved state, conflicts, and safe-mode outcomes</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Hardened live APOD response handling against malformed and unsafe payloads</p>

## Next Steps

Continue to **[Daily Financial Briefing](02_daily_financial_briefing.ipynb)** — you'll review and deploy a canonical multi-API skill, then apply a financial-analyst persona to its evidence.

---

<small>

**Sources:** [NASA APOD API](https://api.nasa.gov/) · [OpenClaw Skills Overview](https://docs.openclaw.ai/tools/skills) · [Creating Skills (SKILL.md Spec)](https://docs.openclaw.ai/tools/creating-skills) · [Network Policy Schema](https://docs.nvidia.com/openshell/reference/policy-schema) · [Policy Customization](https://docs.nvidia.com/openshell/sandboxes/policies) · [Sandbox Management](https://docs.nvidia.com/openshell/sandboxes/manage-sandboxes) · [Agent Workspaces](https://docs.openclaw.ai/concepts/agent-workspace) · [SOUL.md](https://docs.openclaw.ai/concepts/soul) · [NemoClaw Commands](https://docs.nvidia.com/nemoclaw/latest/user-guide/openclaw/reference/commands) · [NemoClaw Agent Skills](https://docs.nvidia.com/nemoclaw/latest/user-guide/openclaw/resources/agent-skills) · [Security Best Practices](https://docs.nvidia.com/openshell/security/best-practices) · [Inference Routing / Privacy Router](https://docs.nvidia.com/openshell/inference/about) · [brevdev/nemoclaw-demos](https://github.com/brevdev/nemoclaw-demos)

</small>